# G1 — 모델 일반화 셀 (W3 고정)

ResNet18 단일 셀서 본 현상이 **다른 CNN에서도 재현되나**:
- **진단 일반화 ①②**: `normHd2`가 단일층 short recovery를 예측하는가? (부분상관, size 통제)
- **적용 일반화 ⑤**: proxy-guided partial QAT가 적은 param%로 full에 근접하는가?

모델 = ResNet34 · MobileNetV2 (CNN 2 family). 모델별 try/except + incremental 저장. **"현상 재현"이지 "selector recipe 제안" 아님.**

In [ ]:
# --- Colab 셋업 ---
import os
REPO = '/content/26_Capstone'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/u-nsiq/26_Capstone.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- 엔진 + Drive + 로더 + 모델 ---
from qat_engine import *
import numpy as np, matplotlib.pyplot as plt, json

try:
    from google.colab import drive; drive.mount('/content/drive')
    ART = '/content/drive/MyDrive/26_Capstone'
except Exception:
    ART = './_local_art'
for sub in ['checkpoints', 'outputs/G1', 'figures']:
    os.makedirs(f'{ART}/{sub}', exist_ok=True)
DATA_ROOT = f'{ART}/data'
OUTDIR = f'{ART}/outputs/G1'
N_BITS = 3

train_loader, val_loader, calib_loader = get_loaders('cifar100', batch=128, calib_size=1024, data_root=DATA_ROOT)

# CIFAR100/W3. ckpt 없으면 train_baseline가 FP 학습(모델별 ~시간 = long pole). ViT는 아래 md 참고.
MODELS = ['resnet34', 'mobilenetv2_100']
print('device', DEVICE, '| models', MODELS)

In [ ]:
# --- G1 실행: 모델별 일반화 셀 (진단①②+적용⑤). incremental 저장 + try/except ---
# 규모: 단일층 sweep(층수 × seed3 × 800) + payoff. ResNet34~37층, MobileNetV2~52층 → 무거움(몇 시간).
#       너무 느리면 seeds=(0,1)·long_t=300으로 줄여도 'same trend'엔 충분.
results = {}
P = f'{OUTDIR}/g1_generalization_w{N_BITS}.json'
for m in MODELS:
    ckpt = f'{ART}/checkpoints/{m}_cifar100_fp32.pt'
    print(f'=== {m} ===  (FP 캐시 없으면 학습)')
    try:
        results[m] = run_generalization_cell(m, train_loader, val_loader, calib_loader, ckpt,
                                             n_bits=N_BITS, B=0.25, short_t=30, long_t=800,
                                             seeds=(0, 1, 2), device=DEVICE)
        json.dump(results, open(P, 'w'), indent=2)        # 모델 하나 끝날 때마다 저장
        r = results[m]; fa = r['fp_acc']; pa = r['ptq_acc']; cs = r['corr_short_partN']; iv = r['inv_1mrho']
        print(f'  OK  fp={fa:.2f} ptq={pa:.2f} | nHd2->short partN={cs:+.2f} | inv 1-rho={iv:+.2f}  (saved)')
    except Exception as e:
        print(f'  FAIL {m}: {type(e).__name__}: {e}')
print('done ->', P)

In [ ]:
# --- 일반화 표 (진단 corr/inv + 적용 payoff) ---
def recov(acc, ptq, full):
    return 100.0 * (acc - ptq) / (full - ptq) if full > ptq else float('nan')
print('model         fp     ptq    gap   | nHd2->short  inv  | nHd2-topk(param%,recov%)  random   full')
print('  resnet18*    76.84  63.36  13.48 |   +0.88      .   | (S1.2/U5 기존 셀)')
for m, r in results.items():
    p5 = r['payoff']; T = r['long_t']
    fa = r['fp_acc']; pa = r['ptq_acc']; g = r['gap']; cs = r['corr_short_partN']; iv = r['inv_1mrho']
    nh_a = p5['acc']['normHd2-topk'][T]; rn_a = p5['acc']['random'][T]; fl_a = p5['acc']['full'][T]
    nh_p = p5['param_ratio']['normHd2-topk'] * 100; rv = recov(nh_a, pa, fl_a)
    print(f'  {m:<12} {fa:5.2f}  {pa:5.2f}  {g:5.2f} |   {cs:+.2f}     {iv:+.2f} | {nh_a:5.2f}({nh_p:.0f}%,{rv:.0f}%)  {rn_a:5.2f}  {fl_a:5.2f}')

In [ ]:
# --- Proxy 일반화 그림: 모델별 normHd2->short 부분상관 ---
names = ['resnet18*'] + list(results.keys())
corrs = [0.88] + [results[m]['corr_short_partN'] for m in results]
plt.figure(figsize=(5, 3.2))
plt.bar(range(len(names)), corrs, color='steelblue'); plt.axhline(0, color='k', lw=0.5)
plt.xticks(range(len(names)), names, rotation=15); plt.ylim(0, 1)
plt.ylabel('partial spearman (normHd2 -> short R)'); plt.title(f'단기proxy 일반화 (W{N_BITS})')
plt.tight_layout(); plt.savefig(f'{OUTDIR}/g1_proxy_corr.png', dpi=120); plt.show()

## G1 해석 (모델 일반화)

**발표 슬라이드 2장:**
1. **일반화 표** — 모델별 PTQ → partial QAT(normHd2-topk) → full, param%·recovery%
2. **Proxy 일반화 그림** — 모델별 normHd2→short 부분상관 (ResNet18만의 결과 아님)

읽는 법:
- **`nHd2->short partN` 양수·큼** → 단기proxy가 다른 CNN서도 회복층 예측 → **진단 일반화** ✓
- **`normHd2-topk`가 적은 param%로 full 근접** → partial QAT payoff 재현 → **적용 일반화** ✓
- **`inv 1-rho` 양수** → 역전(단일층 short-opt ≠ long-opt)도 재현

⚠️ "현상·payoff *재현*"이지 "우리 selector recipe 제안" 아님 (de-scope lock).

### ViT는 왜 여기 없나 (정직)
quant 엔진은 ViT의 `Linear`도 양자화함(`ptq`가 Conv+Linear 둘 다 swap). 근데 **FP 학습 파이프라인(`train_baseline`)이 ResNet식 SGD(lr0.1·60ep)** 라 ViT엔 부적합 — ViT는 AdamW·warmup·224입력·pretrained-finetune 필요. 그냥 넣으면 FP가 부실 학습돼 ViT 결과가 무의미해짐. → **ViT 일반화는 별도 ViT-튜닝 학습으로(후속). 6/19엔 "ViT 확장 예정"으로.**

*트림: 너무 느리면 cell 4의 `seeds=(0,1)`·`long_t=300`. MobileNetV2(~52층)가 ResNet34(~37층)보다 무거움.*